# 00 · Setup — Catálogo, esquemas y volumen
### Proyecto ETL FIFA 21 · Jorge Amat · David Plaza

Este notebook prepara el entorno de Unity Catalog: catálogo, esquemas y volumen. No toca datos — la lectura del CSV y la creación de la tabla Bronze están en `carga_delta.ipynb`.

**Orden de ejecución del proyecto completo:**
1. `00_setup_catalogo` ← este notebook
2. `carga_delta` (lee el CSV del volumen y construye Bronze)
3. `transformaciones` (Silver + Data Quality)
4. `MODELLING` (Gold)


## 1. Catálogo

> ⚠️ Si tu usuario no tiene permisos para crear catálogos, usa uno ya existente y sustituye `fifa_catalog` por ese nombre en todas las celdas de este notebook y de los otros tres.

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS fifa_catalog")
print("✅ Catálogo fifa_catalog listo")

✅ Catálogo fifa_catalog listo


## 2. Esquemas (Bronze / Silver / Gold)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS fifa_catalog.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS fifa_catalog.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS fifa_catalog.gold")

print("✅ Esquemas creados: bronze, silver, gold")

✅ Esquemas creados: bronze, silver, gold


In [0]:
# Verificación rápida: deberían aparecer los 3 esquemas
display(spark.sql("SHOW SCHEMAS IN fifa_catalog"))

databaseName
bronze
default
gold
information_schema
silver


## 3. Volumen para el CSV en bruto

Un Volume es simple almacenamiento de ficheros dentro de Unity Catalog — aquí NO se parsea nada, solo guarda el archivo tal cual, así que es 100% seguro para subir el CSV.

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS fifa_catalog.bronze.landing")
print("✅ Volumen listo: /Volumes/fifa_catalog/bronze/landing")

✅ Volumen listo: /Volumes/fifa_catalog/bronze/landing


## 4. Subir el CSV al volumen (paso manual)

1. Ve a **Catalog Explorer** → `fifa_catalog` → `bronze` → `landing` (el volumen que acabas de crear).
2. **Upload to this volume** → sube el CSV de FIFA 21 tal cual, sin pasar por ningún asistente de "Create Table".
3. Comprueba el nombre exacto del archivo que te queda (normalmente `fifa21_raw_data.csv` o similar) y ajústalo en la celda de abajo si no coincide.

La celda siguiente comprueba si ya lo subiste.

In [0]:
CSV_PATH = "/Volumes/fifa_catalog/bronze/landing/fifa21_raw_data.csv"

import os
existe = os.path.exists(CSV_PATH)
print(f"¿Existe el CSV en {CSV_PATH}? -> {existe}")

if not existe:
    print("⚠️  Sube el archivo al volumen (paso anterior) y, si tiene otro nombre, ajusta CSV_PATH arriba.")
    print("Archivos que sí hay en el volumen ahora mismo:")
    try:
        for f_ in dbutils.fs.ls("/Volumes/fifa_catalog/bronze/landing"):
            print(" -", f_.name)
    except Exception as e:
        print("(volumen vacío o no accesible todavía)")

¿Existe el CSV en /Volumes/fifa_catalog/bronze/landing/fifa21_raw_data.csv? -> True


## Siguiente paso

Con el catálogo, los esquemas y el volumen listos, sube el CSV al volumen (paso 4) y continúa con:

`carga_delta.ipynb` → `transformaciones.ipynb` → `MODELLING.ipynb`
